# Sector Rotation System — CPCV Signal Validation

This notebook implements **Combinatorial Purged Cross-Validation (CPCV)**
to validate that the regime-based trading signals are not overfit to
in-sample data.

**Why CPCV?**
Standard k-fold CV leaks future information into training sets when applied
to time series. CPCV:
- Splits data into non-overlapping groups
- Purges observations near train/test boundaries to prevent leakage
- Applies an embargo period to further reduce autocorrelation contamination
- Tests all unique train/test combinations for statistical robustness

Reference: de Prado, M. L. (2018). *Advances in Financial Machine Learning*.

**Runtime:** ~3–5 minutes on Colab free tier.

## 1. Clone Repo & Install Dependencies

In [ ]:
# --- Clone private repo using Colab Secrets ---
# Store your GitHub PAT in Colab Secrets (key icon in sidebar) as GITHUB_TOKEN.
# The token needs 'repo' scope for private repository access.

import os
from pathlib import Path

REPO_DIR = '/content/sector-rotation-system'

if not os.path.isdir(REPO_DIR):
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
    !git clone https://{token}@github.com/bigbathtub101/sector-rotation-system.git {REPO_DIR}
else:
    print(f'Repo already cloned at {REPO_DIR}, pulling latest...')
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')

# Install dependencies
!pip install -q -r requirements.txt
print('\nSetup complete.')

## 2. Set API Keys & Load Config

In [ ]:
import json
import sqlite3
import datetime as dt
import inspect
from itertools import combinations

import numpy as np
import pandas as pd
import yaml

# Optional: set your FRED API key (free at https://fred.stlouisfed.org)
# os.environ['FRED_API_KEY'] = 'your_key_here'

with open('config.yaml', 'r') as f:
    config = yaml.safe_load(f)

MP_DECAY = config['factor_model']['mclean_pontiff_decay']
MP_LABEL = config['backtest']['mclean_pontiff_label']

# ----- CRITICAL: Define a single DB path used by EVERYTHING -----
DB_PATH = Path(os.getcwd()) / 'rotation_system.db'

print(f"McLean-Pontiff decay factor: {MP_DECAY}")
print(f"Label: {MP_LABEL}")
print(f"DB path: {DB_PATH}")


# --- Helper: patch a module's DB_PATH everywhere it matters ---
def patch_db_path(mod, new_path):
    """
    Overwrite a module's DB_PATH global, then walk its public functions
    and replace any default argument that was the OLD DB_PATH value
    (captured at import time) with the new path.
    """
    old_path = getattr(mod, 'DB_PATH', None)
    mod.DB_PATH = new_path

    for name, obj in inspect.getmembers(mod, inspect.isfunction):
        if obj.__defaults__:
            new_defaults = []
            changed = False
            for d in obj.__defaults__:
                if isinstance(d, Path) and old_path and d == old_path:
                    new_defaults.append(new_path)
                    changed = True
                else:
                    new_defaults.append(d)
            if changed:
                obj.__defaults__ = tuple(new_defaults)

    print(f"  {mod.__name__}.DB_PATH = {mod.DB_PATH}")

print("patch_db_path helper defined.")

## 3. Populate Database (if needed)

If you already ran the `setup_and_backtest.ipynb` notebook and the
database is populated, this cell will skip ingestion. Otherwise it
runs the full data pipeline.

In [ ]:
import data_feeds
import regime_detector

# Patch DB_PATH in both modules (fixes default-argument capture issue)
patch_db_path(data_feeds, DB_PATH)
patch_db_path(regime_detector, DB_PATH)

# Check if DB already has data
conn = sqlite3.connect(str(DB_PATH))
try:
    price_count = pd.read_sql('SELECT COUNT(*) AS n FROM prices', conn).iloc[0]['n']
    signal_count = pd.read_sql('SELECT COUNT(*) AS n FROM signals', conn).iloc[0]['n']
except Exception:
    price_count, signal_count = 0, 0
conn.close()

if price_count < 100:
    print('Database is empty or too small — running full data ingestion...')
    ingestion_result = data_feeds.run_full_ingestion(config)
    print(f"  Prices ingested: {ingestion_result.get('prices', {}).get('rows', 'N/A')} rows")
else:
    print(f'Database already has {price_count:,} price rows — skipping ingestion.')

if signal_count < 10:
    print('Running regime detection to populate signals...')
    regime_result = regime_detector.run_regime_detection(config)
    print(f"  Regime: {regime_result.get('dominant_regime', 'N/A')}")
else:
    print(f'Database already has {signal_count:,} signal rows — skipping regime detection.')

## 4. Load Data

In [ ]:
conn = sqlite3.connect(str(DB_PATH))

# Load SPY benchmark returns
spy = pd.read_sql(
    "SELECT date, close FROM prices WHERE ticker='SPY' ORDER BY date",
    conn, parse_dates=['date']
).set_index('date')
spy_ret = spy['close'].pct_change().dropna()

# Load regime signals
signals = pd.read_sql(
    "SELECT date, signal_data FROM signals WHERE signal_type='regime_state' ORDER BY date",
    conn, parse_dates=['date']
)
conn.close()

# Parse regime (key is 'dominant_regime' in our signal_data JSON)
signals['regime'] = signals['signal_data'].apply(
    lambda x: json.loads(x).get('dominant_regime', 'offense') if isinstance(x, str) else 'offense'
)
signals = signals.set_index('date')

# Merge
data = spy_ret.to_frame('ret').join(signals['regime'], how='left')
data['regime'] = data['regime'].ffill().fillna('offense')

regime_exposure = {'offense': 0.75, 'defense': 0.35, 'panic': 0.05}
data['exposure'] = data['regime'].map(regime_exposure)
data['strategy_ret'] = data['ret'] * data['exposure']

print(f"Total observations: {len(data)}")
print(f"Date range: {data.index[0].strftime('%Y-%m-%d')} to {data.index[-1].strftime('%Y-%m-%d')}")
print(f"\nRegime distribution:")
print(data['regime'].value_counts())

## 5. CPCV Implementation

In [ ]:
def cpcv_split(n_obs: int, n_groups: int = 6, n_test_groups: int = 2,
               purge_days: int = 5, embargo_days: int = 3):
    """
    Generate Combinatorial Purged Cross-Validation splits.
    
    Parameters:
        n_obs:          Total observations
        n_groups:       Number of non-overlapping groups
        n_test_groups:  Number of groups used for testing in each split
        purge_days:     Observations to remove at train/test boundary
        embargo_days:   Additional observations to skip after test set
    
    Returns:
        List of (train_indices, test_indices) tuples
    """
    group_size = n_obs // n_groups
    groups = []
    for i in range(n_groups):
        start = i * group_size
        end = (i + 1) * group_size if i < n_groups - 1 else n_obs
        groups.append(np.arange(start, end))
    
    splits = []
    for test_combo in combinations(range(n_groups), n_test_groups):
        test_indices = np.concatenate([groups[i] for i in test_combo])
        train_groups = [i for i in range(n_groups) if i not in test_combo]
        train_indices = np.concatenate([groups[i] for i in train_groups])
        
        # Purge: remove train observations within purge_days of any test boundary
        test_set = set(test_indices)
        purge_set = set()
        for idx in test_indices:
            for p in range(1, purge_days + 1):
                purge_set.add(idx - p)
                purge_set.add(idx + p)
        
        # Embargo: remove train observations in embargo window after test end
        test_end = int(test_indices.max())
        embargo_set = set(range(test_end + 1, min(test_end + embargo_days + 1, n_obs)))
        
        # Remove purged and embargoed from train
        remove = (purge_set | embargo_set) - test_set
        train_clean = np.array([i for i in train_indices if i not in remove])
        
        splits.append((train_clean, test_indices))
    
    return splits


# Generate splits
N_GROUPS = 6
N_TEST = 2
PURGE = 5
EMBARGO = 3

splits = cpcv_split(len(data), n_groups=N_GROUPS, n_test_groups=N_TEST,
                    purge_days=PURGE, embargo_days=EMBARGO)

n_splits = len(splits)
print(f"CPCV Configuration:")
print(f"  Groups: {N_GROUPS}, Test groups per split: {N_TEST}")
print(f"  Purge window: {PURGE} days, Embargo: {EMBARGO} days")
print(f"  Total splits: {n_splits} (C({N_GROUPS},{N_TEST}))")
print(f"  Avg train size: {np.mean([len(s[0]) for s in splits]):.0f}")
print(f"  Avg test size: {np.mean([len(s[1]) for s in splits]):.0f}")

## 6. Run CPCV Backtest

In [ ]:
def compute_metrics(returns: pd.Series, label: str = ""):
    """Compute annualized return, volatility, Sharpe, max drawdown."""
    if len(returns) < 10:
        return {'ann_ret': 0, 'ann_vol': 0, 'sharpe': 0, 'max_dd': 0}
    
    cum = (1 + returns).cumprod()
    total_ret = cum.iloc[-1] - 1
    days = len(returns)
    ann_factor = 252 / days if days > 0 else 1
    
    ann_ret = (1 + total_ret) ** ann_factor - 1
    ann_vol = returns.std() * np.sqrt(252)
    sharpe = ann_ret / ann_vol if ann_vol > 0 else 0
    
    peak = cum.cummax()
    max_dd = ((cum - peak) / peak).min()
    
    return {'ann_ret': ann_ret, 'ann_vol': ann_vol, 'sharpe': sharpe, 'max_dd': max_dd}


results = []

for i, (train_idx, test_idx) in enumerate(splits):
    test_data = data.iloc[test_idx]
    
    # Strategy returns on the test set (using regime signals as-is)
    strat_metrics = compute_metrics(test_data['strategy_ret'], f"Split {i+1} Strategy")
    bench_metrics = compute_metrics(test_data['ret'], f"Split {i+1} Benchmark")
    
    results.append({
        'split': i + 1,
        'test_start': data.index[test_idx[0]].strftime('%Y-%m-%d'),
        'test_end': data.index[test_idx[-1]].strftime('%Y-%m-%d'),
        'test_days': len(test_idx),
        'train_days': len(train_idx),
        'strat_ann_ret': strat_metrics['ann_ret'],
        'strat_sharpe': strat_metrics['sharpe'],
        'strat_max_dd': strat_metrics['max_dd'],
        'bench_ann_ret': bench_metrics['ann_ret'],
        'bench_sharpe': bench_metrics['sharpe'],
        'bench_max_dd': bench_metrics['max_dd'],
        'alpha_raw': strat_metrics['ann_ret'] - bench_metrics['ann_ret'],
        'alpha_adj': (strat_metrics['ann_ret'] - bench_metrics['ann_ret']) * MP_DECAY,
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False, float_format='{:.3f}'.format))

## 7. Aggregate Results & Statistical Tests

In [ ]:
from scipy import stats

print("=" * 70)
print("CPCV AGGREGATE RESULTS")
print("=" * 70)

# Summary statistics
print(f"\n{'Metric':<35} {'Mean':>10} {'Std':>10} {'Min':>10} {'Max':>10}")
print("-" * 75)

for col, label in [('strat_ann_ret', 'Strategy Ann. Return'),
                    ('strat_sharpe', 'Strategy Sharpe'),
                    ('strat_max_dd', 'Strategy Max Drawdown'),
                    ('bench_ann_ret', 'Benchmark Ann. Return'),
                    ('bench_sharpe', 'Benchmark Sharpe'),
                    ('alpha_raw', 'Raw Alpha'),
                    ('alpha_adj', f'Adjusted Alpha (\u00d7{MP_DECAY})')]:
    vals = results_df[col]
    print(f"{label:<35} {vals.mean():>9.3f} {vals.std():>9.3f} {vals.min():>9.3f} {vals.max():>9.3f}")

# t-test: is alpha significantly different from zero?
alpha_vals = results_df['alpha_raw'].values
t_stat, p_value = stats.ttest_1samp(alpha_vals, 0)

print(f"\n{'='*70}")
print(f"STATISTICAL SIGNIFICANCE")
print(f"{'='*70}")
print(f"H0: Mean raw alpha = 0")
print(f"t-statistic:  {t_stat:.3f}")
print(f"p-value:      {p_value:.4f}")
print(f"Significant at 5%: {'YES' if p_value < 0.05 else 'NO'}")
print(f"Significant at 10%: {'YES' if p_value < 0.10 else 'NO'}")

# Fraction of splits with positive alpha
pos_frac = (results_df['alpha_raw'] > 0).mean()
print(f"\nSplits with positive alpha: {pos_frac:.0%} ({int(pos_frac * n_splits)}/{n_splits})")

# Deflated Sharpe approximation
mean_sharpe = results_df['strat_sharpe'].mean()
std_sharpe = results_df['strat_sharpe'].std()
if std_sharpe > 0:
    dsr = stats.norm.cdf((mean_sharpe - 0) / std_sharpe * np.sqrt(n_splits))
    print(f"\nDeflated Sharpe Ratio (DSR) p-value: {1 - dsr:.4f}")
    print(f"  (Probability that observed Sharpe is due to chance)")

print(f"\n{MP_LABEL}")

## 8. Visualize Results

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('CPCV Validation Results', fontsize=14, fontweight='bold')

# 1. Alpha distribution
ax = axes[0, 0]
ax.hist(results_df['alpha_raw'], bins=max(5, n_splits // 3), color='steelblue',
        edgecolor='white', alpha=0.8, label='Raw Alpha')
ax.axvline(0, color='red', linestyle='--', linewidth=1.5, label='Zero')
ax.axvline(results_df['alpha_raw'].mean(), color='green', linestyle='-',
           linewidth=2, label=f'Mean: {results_df["alpha_raw"].mean():.2%}')
ax.set_title('Alpha Distribution (Across Splits)')
ax.set_xlabel('Annualized Alpha')
ax.set_ylabel('Count')
ax.legend(fontsize=9)

# 2. Sharpe comparison
ax = axes[0, 1]
x = np.arange(n_splits)
w = 0.35
ax.bar(x - w/2, results_df['strat_sharpe'], w, label='Strategy', color='steelblue')
ax.bar(x + w/2, results_df['bench_sharpe'], w, label='Benchmark', color='lightcoral')
ax.set_title('Sharpe Ratio by Split')
ax.set_xlabel('Split')
ax.set_ylabel('Sharpe Ratio')
ax.set_xticks(x)
ax.set_xticklabels([f'S{i+1}' for i in x])
ax.legend(fontsize=9)

# 3. Max drawdown comparison
ax = axes[1, 0]
ax.bar(x - w/2, results_df['strat_max_dd'] * 100, w, label='Strategy', color='steelblue')
ax.bar(x + w/2, results_df['bench_max_dd'] * 100, w, label='Benchmark', color='lightcoral')
ax.set_title('Max Drawdown by Split (%)')
ax.set_xlabel('Split')
ax.set_ylabel('Max Drawdown (%)')
ax.set_xticks(x)
ax.set_xticklabels([f'S{i+1}' for i in x])
ax.legend(fontsize=9)

# 4. Adjusted alpha
ax = axes[1, 1]
colors = ['green' if a > 0 else 'red' for a in results_df['alpha_adj']]
ax.bar(x, results_df['alpha_adj'] * 100, color=colors, edgecolor='white')
ax.axhline(0, color='black', linewidth=0.8)
ax.axhline(results_df['alpha_adj'].mean() * 100, color='blue', linestyle='--',
           label=f'Mean: {results_df["alpha_adj"].mean():.2%}')
ax.set_title(f'McLean-Pontiff Adjusted Alpha (\u00d7{MP_DECAY})')
ax.set_xlabel('Split')
ax.set_ylabel('Adjusted Alpha (%)')
ax.set_xticks(x)
ax.set_xticklabels([f'S{i+1}' for i in x])
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('cpcv_validation.png', dpi=150, bbox_inches='tight')
plt.show()
print("Chart saved to cpcv_validation.png")

## 9. Summary & Interpretation

**How to read the results:**

| Outcome | Interpretation |
|---------|--------------|
| Positive mean alpha, p < 0.05 | Strong evidence the signal has real predictive power |
| Positive mean alpha, p < 0.10 | Moderate evidence — worth monitoring, not conclusive |
| Positive mean alpha, p > 0.10 | Weak evidence — alpha may be noise or overfit |
| >75% of splits positive | Consistent signal across market conditions |
| <50% of splits positive | Unreliable — signal works in some regimes but not others |
| Strategy MDD << Benchmark MDD | Risk management (regime-based allocation) is working |

**Key caveats:**
- All alpha figures should be discounted by the McLean-Pontiff factor (0.74) for forward estimates
- With ~2 years of data, statistical power is limited — more data improves confidence
- CPCV mitigates but does not eliminate overfitting risk
- Transaction costs are not included in this simplified backtest